[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kkaitechnology-coder/free-python-for-development/blob/main/lesson-4-2-sql-databases-with-sqlalchemy.ipynb)

# Lesson 4.2 · SQL Databases with SQLAlchemy

**Free Python For Development** · Topic 4: File I/O, Databases & REST APIs

_sqlite3, CRUD, SQLAlchemy ORM, Alembic._

▶ **Watch this lesson with the AI voice tutor:** [open it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-4-2)

**How to use this notebook:** run each code cell with **Shift + Enter**, or use **Runtime → Run all**. Then change the code and run it again — experimenting is the fastest way to learn.

**Setup** — install the libraries this lesson uses (takes a few seconds):

In [ ]:
!pip install -q "sqlalchemy>=2" alembic

## Introduction & Core Concepts of SQL Databases with SQLAlchemy

Let’s talk about one of the most powerful ways Python developers work with data: using **SQL databases**. You have probably heard of MySQL, PostgreSQL or SQLite. These are organized places to store data for your applications — much like sorting files into labeled cabinets instead of leaving them scattered across your desk.

Python ships with the built-in `sqlite3` library, which lets you execute raw SQL. That works for small scripts, but as soon as an application grows, writing SQL strings by hand becomes repetitive and error-prone.

> 💡 **Key insight:** SQLAlchemy is a translator between Python and SQL. Instead of writing SQL queries yourself, you describe tables as Python classes and express operations with ordinary Python objects. This approach is called an **Object-Relational Mapper (ORM)**.

### Why an ORM Matters

- You stay inside the Python type system and tooling you already know.
- SQL injection risks drop dramatically because parameters are handled automatically.
- The same model code can target SQLite in development and PostgreSQL in production.
- Relationships (one-to-many, many-to-many) become simple attributes instead of JOIN strings.

> 💡 **Mental model:** Think of a library. With raw SQL you write instructions like “INSERT INTO books …”. With an ORM you create a `Book` object, set its attributes, and tell the session “please remember this book”. SQLAlchemy turns that request into the correct SQL behind the scenes.

## Architecture & Implementation Deep Dive

SQLAlchemy’s modern style (2.0+) uses declarative models and a `Session` object that tracks changes. The classic workflow is known as **CRUD**: Create, Read, Update, Delete.

**Model**

A Python class that maps to a database table. Columns become class attributes with types.

**Session**

A temporary workspace. You add, modify or delete objects inside it, then call `commit()` to make the changes permanent.

### Defining a Model and Performing CRUD

In [ ]:
from pathlib import Path
Path("library.db").unlink(missing_ok=True)   # start with an empty database on every run

from sqlalchemy import create_engine, String, Integer
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

class Base(DeclarativeBase):
    pass

class Book(Base):
    __tablename__ = "books"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    author: Mapped[str] = mapped_column(String(100))
    pages: Mapped[int] = mapped_column(Integer, default=0)

# SQLite for local development
engine = create_engine("sqlite:///library.db", echo=False)
Base.metadata.create_all(engine)

# ----- CREATE -----
with Session(engine) as session:
    book = Book(title="1984", author="George Orwell", pages=328)
    session.add(book)
    session.commit()
    print("Created:", book.id, book.title)

# ----- READ -----
with Session(engine) as session:
    found = session.get(Book, 1)
    print("Read:", found.title if found else "not found")

# ----- UPDATE -----
with Session(engine) as session:
    book = session.get(Book, 1)
    if book:
        book.pages = 330
        session.commit()
        print("Updated pages to", book.pages)

# ----- DELETE -----
with Session(engine) as session:
    book = session.get(Book, 1)
    if book:
        session.delete(book)
        session.commit()
        print("Deleted")

**Step-by-step**

- Declare a model that inherits from `DeclarativeBase`.
- `create_all` builds the table if it does not exist.
- A `Session` is a short-lived unit of work.
- `add` + `commit` persists a new row.
- `session.get` fetches by primary key.
- Changing an attribute and committing issues an UPDATE.
- `delete` + `commit` removes the row.

> 💡 **Architecture note:** Keep sessions short-lived. Open a session, do the work, commit or roll back, then close it. Long-lived sessions can lead to stale data and hard-to-debug identity-map issues.

## Best Practices & Production Integration — Alembic

Applications evolve. You may need to add a new column, rename a table, or create an index. Changing the database structure by hand is risky. **Alembic** is the official migration tool that works hand-in-hand with SQLAlchemy. It gives you version-controlled schema changes — essentially “git for your database”.

### Typical Alembic Workflow

Alembic writes migration files like the one below, and runs them with `alembic upgrade head` inside a project:

```python
# 1. Install and initialize (run once)
# pip install alembic
# alembic init migrations

# 2. After changing a model, autogenerate a migration
# alembic revision --autogenerate -m "add genre column"

# Example of the generated migration file (simplified)
"""add genre column

Revision ID: a1b2c3d4e5f6
"""
from alembic import op
import sqlalchemy as sa

def upgrade() -> None:
    op.add_column(
        "books",
        sa.Column("genre", sa.String(50), nullable=True)
    )

def downgrade() -> None:
    op.drop_column("books", "genre")

# 3. Apply the migration
# alembic upgrade head

# 4. Roll back if needed
# alembic downgrade -1
```

You can apply the same change right here, using Alembic’s operations API on the SQLite database from above:

In [ ]:
import sqlalchemy as sa
from alembic.migration import MigrationContext
from alembic.operations import Operations

columns = [c["name"] for c in sa.inspect(engine).get_columns("books")]
if "genre" not in columns:
    with engine.begin() as conn:
        op = Operations(MigrationContext.configure(conn))
        op.add_column("books", sa.Column("genre", sa.String(50), nullable=True))

print("books columns:", [c["name"] for c in sa.inspect(engine).get_columns("books")])

**Production habits**

- Never edit the database schema manually in production.
- Review autogenerated migrations before applying them.
- Keep migrations in the same repository as the application code.
- Test both `upgrade` and `downgrade` paths.
- Use environment-specific connection URLs (dev / staging / prod).

**SQLAlchemy strengths**

Readable Python models, automatic parameter binding, multi-database support, rich relationship handling, and a powerful expression language for complex queries.

**When to drop to raw SQL**

Extremely complex analytical queries, vendor-specific features, or bulk operations where the ORM overhead is measurable. SQLAlchemy still lets you execute raw SQL safely when needed.

> 💡 **Big takeaway:** SQLAlchemy + Alembic give you the reliability of SQL together with the readability and maintainability of Python. You describe your data as classes, perform CRUD through ordinary objects, and evolve the schema safely with versioned migrations — exactly what production applications need.

---

✅ **Lesson complete!** Next up: **Lesson 4.3 · MongoDB with PyMongo** — [watch it in OratorLab](https://oratorlab.in/course-player.html?courseId=python-for-development&subtopicId=subtopic-4-3) or open `lesson-4-3-mongodb-with-pymongo.ipynb`.

Learn the full **Free Python For Development** course for free at [oratorlab.in](https://oratorlab.in/courses/python-for-development).